# Sawti — Phase 5 step 3: QLoRA fine-tuning

Runs `scripts/train_qlora.py` on a CUDA Colab runtime. This machine's own dev environment has no CUDA (`bitsandbytes` needs a real GPU), so this notebook, not a local run, is how the training in `docs/09-DECISIONS.md` (2026-09-25, "Phase 5 fine-tuning dataset" and the QLoRA hyperparameter entry) actually executes.

**Before running:** Runtime → Change runtime type → **T4 GPU** (Colab's free tier). The estimate in `scripts/train_qlora.py`'s own docstring — ~10-15 minutes, dominated by downloading/quantizing the ~16 GB base checkpoint rather than by the 18-example training set itself — is unmeasured, so watch the first logged training step before assuming the rest will finish cleanly.

**What this trains on is synthetic.** Every example in `data/finetune/train.jsonl`/`val.jsonl` is manufactured by diffing an agent output against an LLM-generated reference label — never a real QA reviewer's judgment. See `scripts/build_finetune_dataset.py`'s docstring and `docs/09-DECISIONS.md`. This run validates the QLoRA mechanism end-to-end; it is not evidence of real-world learning capacity.

## 1. Confirm the GPU

In [ ]:
!nvidia-smi

## 2. Mount Drive (optional, for persisting the adapter past this session)

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/sawti-phase5"  # adjust if you want a different Drive path
!mkdir -p "$DRIVE_OUTPUT_DIR"

## 3. Get the repo and the fine-tuning dataset

Two options — use whichever fits. Only `data/finetune/{train,val}.jsonl` is strictly required by `scripts/train_qlora.py`; the rest of the repo is needed for `sawti.config` and the script itself.

In [ ]:
# Option A: clone the repo (needs Colab to have access — a public repo, or
# a token/deploy key for a private one). Replace the URL with this project's
# actual remote.
!git clone https://github.com/<owner>/<repo>.git sawti
%cd sawti

In [ ]:
# Option B (use instead of A if the repo isn't reachable from Colab): upload
# data/finetune/train.jsonl and data/finetune/val.jsonl directly, and pull
# scripts/train_qlora.py + src/sawti/config.py the same way, preserving the
# same relative paths under ./sawti/.
#
# from google.colab import files
# uploaded = files.upload()  # then move the uploaded files into place

## 4. Install the `finetune` extra

CUDA-only — this is exactly the install that fails (deliberately) on this project's Mac dev environment. See `pyproject.toml`'s `finetune` group and `docs/09-DECISIONS.md`.

In [ ]:
!pip install -q -e ".[finetune]"

## 5. Train

`SAWTI_FINETUNE_BASE_MODEL` defaults to `Qwen/Qwen3-8B` (`src/sawti/config.py`) — override it here only if you're intentionally training a different checkpoint. Loss (train + eval) is logged per step to `data/finetune/train_log.csv` — no Langfuse/W&B wiring, see `scripts/train_qlora.py`'s docstring for why.

In [ ]:
import os

# os.environ["SAWTI_FINETUNE_BASE_MODEL"] = "Qwen/Qwen3-8B"  # uncomment to override the default

!python scripts/train_qlora.py

## 6. Inspect the loss curve

In [ ]:
import pandas as pd

log = pd.read_csv("data/finetune/train_log.csv")
log

## 7. Copy the adapter + loss log to Drive

In [ ]:
!cp -r data/finetune/qlora_adapter "$DRIVE_OUTPUT_DIR/qlora_adapter"
!cp data/finetune/train_log.csv "$DRIVE_OUTPUT_DIR/train_log.csv"
print("Saved to", DRIVE_OUTPUT_DIR)

## 8. Step 4 — catastrophic-forgetting check

Runs `scripts/check_forgetting.py` in this same session, while the base checkpoint is already downloaded and the adapter is already on disk. A handful of generic (non-call-analysis) prompts, generated by base Qwen3-8B and by the QLoRA-tuned model, compared for a large regression (empty output, or a drastic length collapse) — not a full eval suite. See that script's docstring and `docs/09-DECISIONS.md`.

In [ ]:
!python scripts/check_forgetting.py

In [ ]:
import json

rows = json.load(open("data/finetune/forgetting_eval.json"))
for row in rows:
    marker = "FLAGGED" if row["flagged"] else "ok"
    print(f"[{marker}] {row['prompt']}")
    print("  base :", row["base_response"][:200])
    print("  tuned:", row["tuned_response"][:200])
    print()

In [ ]:
!cp data/finetune/forgetting_eval.json "$DRIVE_OUTPUT_DIR/forgetting_eval.json"
print("Saved to", DRIVE_OUTPUT_DIR)

## After this notebook

Bring the numbers back to the repo, not just Drive: the loss curve (`train_log.csv`) and the forgetting-eval flags belong in `eval_results.md`, dated, and the base-model/hyperparameter choices are already logged in `docs/09-DECISIONS.md` (2026-09-25) — this notebook is what fills in the measured numbers that entry currently marks as estimates.